# Gradient Boost and XGBoost

In this notebook, we will be exploring Gradient Boost and XGBoost using the existing Titanic dataset.

We will follow the basic preprocessing pipeline and train/validation split of the initial titanic.ipynb to ensure consistency.

We will then build a pipeline using *GradientBoostingClassifier*, then build a pipeline using *XGBClassifier*. Validation accuracies of XGB and Gradient Boost will be compared against earlier models to assess how well the new model performed.

This notebook aims to reinforce the learning of gradient boost and XGBoost. 

## 01. Load Data

In [1]:
# Import Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Set Up Complete")

# Load Data
titanic = pd.read_csv("train.csv")
titanic.head()

Set Up Complete


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 02. Train/validation split, Basic Preprocessing Pipeline

Drop Passenger ID, Name, Ticket, Cabin columns
Separate X and y
Train/validation split
Fit preprocessing pipeline only on X_train

In [2]:
# Drop Passenger ID, Name, Ticket, Cabin
titanic_v2 = titanic.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Train/Test Split
from sklearn.model_selection import train_test_split
X = titanic_v2.drop(columns=["Survived"])
y = titanic_v2["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate numerical and categorical columns
numerical_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]

# Import necessary libraries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns),
                                   ("cat", categorical_transformer, categorical_columns)],
                                   remainder = 'passthrough')

## 03. Gradient Boosting Pipeline

Random Forest builds many trees independently and combines their predictions, while Gradient Boost builds trees sequentially, each tree correcting the errors made by the existing ensemble. 

We will first keep the default settings to establish a baseline before experimenting with other parameters and hyperparameters.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Gradient Boosting Pipeline
gradient_boosting = Pipeline([("preprocessing", preprocessing), ("classifier", GradientBoostingClassifier(random_state=42))])

# Train model
gradient_boosting.fit(X_train, y_train)

# Predict on validation data
gradient_boosting_y_pred = gradient_boosting.predict(X_valid)

# Evaluate accuracy
accuracy_gradient_boosting = accuracy_score(y_valid, gradient_boosting_y_pred)
print(f"Gradient Boosting Accuracy: {accuracy_gradient_boosting:.4f}")

Gradient Boosting Accuracy: 0.8212


### 3.1 Experiment with n_estimators

In Gradient Boost, n_estimators controls the number of boosting stages. The default is 100, so we will explore different number of trees and check their validation accuracy scores.

From 50 to 200, adding boosting stages generally improves validation accuracy. Beyond 200, accuracy decreased slightly then levelled off for this particular validation test split.

In [7]:
n_estimator_values = [50, 100, 150, 200, 250, 300]
gradient_boosting_accuracies = []

for n in n_estimator_values:
    gradient_boosting = Pipeline([("preprocessing", preprocessing), ("classifier", GradientBoostingClassifier(n_estimators=n, random_state=42))])
    gradient_boosting.fit(X_train, y_train)
    gradient_boosting_y_pred = gradient_boosting.predict(X_valid)
    accuracy = accuracy_score(y_valid, gradient_boosting_y_pred)
    gradient_boosting_accuracies.append(accuracy)

for n, accuracy in zip(n_estimator_values, gradient_boosting_accuracies):
    print(f"n_estimators={n}: accuracy={accuracy:.4f}")


n_estimators=50: accuracy=0.7989
n_estimators=100: accuracy=0.8212
n_estimators=150: accuracy=0.8212
n_estimators=200: accuracy=0.8324
n_estimators=250: accuracy=0.8268
n_estimators=300: accuracy=0.8268


## 04. XGBoost

In Gradient Boost, each new tree helps correct errors from the existing ensemble. XGBoost builds on gradient boosting with additional regularisation and optimisation techniques designed to improve training efficiency and model performance. 

In [8]:
from xgboost import XGBClassifier

# Build XGB Pipeline
xgb_model = Pipeline([("preprocessing", preprocessing), ("classifier", XGBClassifier(random_state=42, eval_metric="logloss"))])

# Train model
xgb_model.fit(X_train, y_train)

# Predict on validation data
xgb_y_pred = xgb_model.predict(X_valid)

#Evaluate accuracy
accuracy_xgb = accuracy_score(y_valid, xgb_y_pred)
print(f"XGBoost Accuracy: {accuracy_xgb:.4f}")

XGBoost Accuracy: 0.7933


### 4.1 Experiment with n_estimators

In [10]:
n_estimator_values = [10, 50, 100, 150, 200, 250, 300]
xgb_accuracies = []

for n in n_estimator_values:
    xgb_model = Pipeline([("preprocessing", preprocessing), ("classifier", XGBClassifier(n_estimators=n, random_state=42, eval_metric="logloss"))])
    xgb_model.fit(X_train, y_train)
    xgb_y_pred = xgb_model.predict(X_valid)
    accuracy = accuracy_score(y_valid, xgb_y_pred)
    xgb_accuracies.append(accuracy)

for n, accuracy in zip(n_estimator_values, xgb_accuracies):
    print(f"n_estimators={n}: accuracy={accuracy:.4f}")    

n_estimators=10: accuracy=0.8268
n_estimators=50: accuracy=0.8101
n_estimators=100: accuracy=0.7933
n_estimators=150: accuracy=0.7877
n_estimators=200: accuracy=0.7877
n_estimators=250: accuracy=0.7709
n_estimators=300: accuracy=0.7765


### 4.2 Experiment with max_depth

Keep n_estimators = 10 as it gave us the highest accuracy earlier.

In [14]:
# Try different max_depth values
max_depth_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
xgboost_depth_accuracies = []

for depth in max_depth_values:
    xgboost_model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", XGBClassifier(
            n_estimators=10,
            max_depth=depth,
            random_state=42,
            eval_metric="logloss"
        ))
    ])

    xgboost_model.fit(X_train, y_train)
    xgboost_y_pred = xgboost_model.predict(X_valid)

    accuracy = accuracy_score(y_valid, xgboost_y_pred)
    xgboost_depth_accuracies.append(accuracy)

for depth, accuracy in zip(
    max_depth_values, xgboost_depth_accuracies
):
    print(f"max_depth={depth}: accuracy={accuracy:.4f}")

max_depth=1: accuracy=0.7821
max_depth=2: accuracy=0.8101
max_depth=3: accuracy=0.8156
max_depth=4: accuracy=0.8045
max_depth=5: accuracy=0.8268
max_depth=6: accuracy=0.8268
max_depth=7: accuracy=0.8492
max_depth=8: accuracy=0.8436
max_depth=9: accuracy=0.8268
max_depth=10: accuracy=0.8268


### 4.3 Experiment with learning_rates

Keep n_estimators = 10 and max_depth = 7 as it gave us the highest accuracy earlier.

In [18]:
# Try different learning_rates
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
xgboost_learningrate_accuracies = []

for learning_rate in learning_rates:
    xgboost_model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", XGBClassifier(
            n_estimators=10,
            max_depth=7,
            learning_rate=learning_rate,
            random_state=42,
            eval_metric="logloss"
        ))
    ])

    xgboost_model.fit(X_train, y_train)
    xgboost_y_pred = xgboost_model.predict(X_valid)

    accuracy = accuracy_score(y_valid, xgboost_y_pred)
    xgboost_learningrate_accuracies.append(accuracy)

for learningrate, accuracy in zip(learning_rates, xgboost_learningrate_accuracies):
    print(f"learning rate={learningrate}: accuracy={accuracy:.4f}")

learning rate=0.01: accuracy=0.5866
learning rate=0.05: accuracy=0.8156
learning rate=0.1: accuracy=0.8156
learning rate=0.2: accuracy=0.8380
learning rate=0.3: accuracy=0.8492
learning rate=0.4: accuracy=0.8436
learning rate=0.5: accuracy=0.8324


## 05. Compare results

Using Gradient Boost with its default settings gave a validation accuracy of 0.8212. Increasing n_estimators to 200 improved the validation accuracy to 0.8324, the best result among the Gradient Boosting configurations tested. 

Using XGBoost with its default settings gave a validation accuracy of 0.7933. Reducing n_estimators to 10 improved the validation accuracy to 0.8268. Keeping n_estimators=10 and changing max_depth to 7 further improved the validation accuracy to 0.8492. Finally, adjusting learning_rate to 0.3 maintained the validation accuracy at 0.8492. 

Among the configurations tested, XGBoost with n_estimators=10, max_depth=7, learning_rate=0.3 achieved the highest validation accuracy of 0.8492, outperforming the Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting models on this valdiation split. However, since the models were evaluated using a single train-validation split, this result does not necessarily mean XGBoost will perform best on unseen data. 

## 06. Submission

Submit XGB with n_estimator=10, max_depth=7, and learning_rate=0.3 with highest accuracy of 0.8492.

In [19]:
# Load test.csv
test = pd.read_csv("test.csv")

# Drop columns but keep PassengerID
test_passenger_id = test["PassengerId"]
test = test.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])


# Predict XGB n_estimator=10, max_depth=7, learning_rate=0.3
xgb = Pipeline([("preprocessing", preprocessing),
                  ("classifier", XGBClassifier(n_estimators=10, max_depth=7, learning_rate=0.3, random_state=42, eval_metric="logloss"))])
xgb.fit(X_train, y_train)
xgb_test_pred = xgb.predict(test)
submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_test_pred})
submission.head()
submission.to_csv("titanic_submission_v3_xgb.csv", index=False)
submission.shape

(418, 2)

## 07. Conclusion

| Model | Validation Accuracy | Kaggle score |
| ----- | ------------------- | -----------: |
| Logistic Regression |0.8101 |0.76315|
| Random Forest, `n_estimators=1`|0.8268|0.67942|
| Random Forest, `n_estimators=100` |0.8045|0.75119|
| XGBoost|0.8492|**0.77751**|

XGBoost scored 0.8492 on the validation split, but 0.77751 on Kaggle. As the scores were calculated on different sets of passengers, so this difference is not unusual.

XGBoost submission beat Logistic Regression by 1.44 percentage points, showing an improvement to the model.